In [0]:
#%pip install Faker

In [0]:
# import libraries
from faker import Faker
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [0]:
fake = Faker()

def generate_initial_records(n=1000):
    """Initial load — everything is an INSERT"""
    records = []
    for _ in range(n):
        records.append({
            "customer_id": fake.uuid4(),
            "name": fake.name(),
            "email": fake.email() if np.random.random() > 0.1 else None,
            "phone": fake.phone_number() if np.random.random() > 0.15 else None,
            "city": fake.city(),
            "purchase_amount": round(fake.random.uniform(10, 500), 2),
            "signup_date": str(fake.date_between(start_date="-2y", end_date="today")),
            "_change_type": "INSERT",
            "_change_timestamp": fake.date_time_between(start_date="-2y", end_date="-1d")
        })
    return records


def generate_cdc_batch(existing_records, n_changes=200):
    """
    Simulate a batch of changes arriving later:
    - Some existing customers update their email or city
    - Some existing customers get deleted
    - Some brand new customers arrive as inserts
    """
    changes = []
    existing_df = pd.DataFrame(existing_records)

    # ~40% of changes are UPDATEs to existing customers
    n_updates = int(n_changes * 0.4)
    updated_customers = existing_df.sample(n=n_updates).to_dict("records")
    for row in updated_customers:
        row = row.copy()
        row["email"] = fake.email()           # they changed their email
        row["city"] = fake.city()             # or moved city
        row["_change_type"] = "UPDATE"
        row["_change_timestamp"] = datetime.now() - timedelta(hours=np.random.randint(1, 48))
        changes.append(row)

    # ~10% of changes are DELETEs
    n_deletes = int(n_changes * 0.1)
    deleted_customers = existing_df.sample(n=n_deletes).to_dict("records")
    for row in deleted_customers:
        row = row.copy()
        row["_change_type"] = "DELETE"
        row["_change_timestamp"] = datetime.now() - timedelta(hours=np.random.randint(1, 12))
        changes.append(row)

    # ~50% are new INSERT customers
    n_inserts = n_changes - n_updates - n_deletes
    for _ in range(n_inserts):
        changes.append({
            "customer_id": fake.uuid4(),
            "name": fake.name(),
            "email": fake.email() if np.random.random() > 0.1 else None,
            "phone": fake.phone_number() if np.random.random() > 0.15 else None,
            "city": fake.city(),
            "purchase_amount": round(fake.random.uniform(10, 500), 2),
            "signup_date": str(fake.date_between(start_date="-6m", end_date="today")),
            "_change_type": "INSERT",
            "_change_timestamp": datetime.now() - timedelta(hours=np.random.randint(1, 6))
        })

    return changes

In [0]:
# Generate and save initial records
initial_records = generate_initial_records(1000)
initial_df = pd.DataFrame(initial_records)
bronze_df = spark.createDataFrame(initial_df)

In [0]:
# Save as your CDC source table (raw incoming events)
bronze_df.write.format("delta").mode("overwrite").saveAsTable("databees.bronze.customer_events")

print(f"Initial load: {bronze_df.count()} rows")
bronze_df.groupBy("_change_type").count().show()

In [0]:
cdc_batch = generate_cdc_batch(initial_records)
cdc_batch_df = spark.createDataFrame(pd.DataFrame(cdc_batch))
# Save as your CDC target table (incremental changes)
cdc_batch_df.write.format("delta").mode("append").saveAsTable("databees.bronze.customer_events")
print(f"CDC batch: {cdc_batch_df.count()} rows")
cdc_batch_df.groupBy("_change_type").count().show()